## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Sentence Window Retrieval — LlamaIndex for sentence parsing and window-expanded retrieval, OpenAI for embeddings, Anthropic for answer synthesis.
**Expected output**: Setup confirmation with model names, window size, and LlamaIndex version.

In [ ]:
%pip install -q llama-index llama-index-embeddings-openai llama-index-llms-anthropic python-dotenv

import os
import time
import pathlib

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

# LlamaIndex core
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.schema import Document as LlamaDocument
from llama_index.embeddings.openai import OpenAIEmbedding

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

ANSWER_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL  = 'text-embedding-3-small'
WINDOW_SIZE  = 3   # ±3 sentences around each matched sentence
K_RETRIEVE   = 3   # top-k sentence nodes to retrieve

# Configure LlamaIndex to use OpenAI embeddings; disable built-in LLM
# (generation is handled by the Anthropic SDK directly)
Settings.embed_model = OpenAIEmbedding(model=EMBED_MODEL)
Settings.llm = None

client = Anthropic()

print('Setup complete')
print(f'  Answer model : {ANSWER_MODEL}')
print(f'  Embed model  : {EMBED_MODEL}')
print(f'  Window size  : ±{WINDOW_SIZE} sentences')
print(f'  Retrieve k   : {K_RETRIEVE} sentence nodes')

## Cell 2: Data — ISDA Master Agreement Excerpt
**What this demonstrates**: Loading an ISDA agreement — a dense legal document where individual sentences define obligations, triggers, and conditions. This is an ideal corpus for Sentence Window Retrieval: the most relevant information is a single sentence (e.g., a margin call trigger), but the surrounding sentences provide the definitional context needed for a complete answer.
**Expected output**: Sentence count, word count, and a preview of why sentence-level precision matters in this document type.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

text = (BASE_DIR / 'isda_excerpt.txt').read_text(encoding='utf-8')

# Wrap in a LlamaDocument — the unit SentenceWindowNodeParser operates on
raw_doc = LlamaDocument(text=text, metadata={'source': 'isda_excerpt'})

# Quick sentence count estimate (split on '. ')
approx_sentences = len([s for s in text.split('. ') if len(s.strip()) > 20])

print(f'Loaded: {len(text):,} characters, ~{approx_sentences} sentences')
print()
print('Why Sentence Window matters for ISDA text:')
print('  Every sentence may be a legally distinct obligation, trigger, or condition.')
print('  A margin call trigger is one sentence; surrounding sentences define')
print('  "Threshold Amount", "Credit Support Amount", and the cure period.')
print('  Standard chunking embeds all of these together, diluting the match.')
print('  Sentence Window isolates the trigger sentence, then expands for context.')

## Cell 3: Core — Sentence Parser, Index, Window Expansion, Pipeline
**What this demonstrates**: Two functions implement the full pattern: `build_sentence_index` parses the document into sentence nodes (storing the ±k window as metadata), builds the vector index, and returns the retriever and postprocessor. `sentence_window_rag` runs query → retrieve sentences → expand windows → generate.
**Expected output**: Index built with sentence node count, pipeline functions confirmed.

In [ ]:
def build_sentence_index(
    doc: LlamaDocument,
    window_size: int = WINDOW_SIZE,
) -> tuple:
    """Parse document into sentence nodes and build a sentence-level vector index.

    SentenceWindowNodeParser splits the document into individual sentences.
    Each sentence node stores the ±window_size surrounding sentences in its
    metadata under the key 'window' — this is used at retrieval time to expand
    the matched sentence back to its context.

    Args:
        doc:         LlamaDocument to parse and index.
        window_size: Number of surrounding sentences to store in each node's metadata.

    Returns:
        Tuple of (retriever, postprocessor, node_count).
    """
    # Parse document into sentence nodes; each node embeds one sentence
    # but stores ±window_size sentences in metadata['window']
    parser = SentenceWindowNodeParser.from_defaults(
        window_size=window_size,
        window_metadata_key='window',       # key for the stored window text
        original_text_metadata_key='original_text',  # preserve original sentence
    )
    nodes = parser.get_nodes_from_documents([doc])

    # Build sentence-level vector index — each node's embedding is its sentence
    index = VectorStoreIndex(nodes)

    # Retriever returns top-k sentence nodes by embedding similarity
    retriever = index.as_retriever(similarity_top_k=K_RETRIEVE)

    # Postprocessor replaces each matched sentence with its stored window
    # at generation time — retrieval precision, window-level context
    postprocessor = MetadataReplacementPostProcessor(target_metadata_key='window')

    return retriever, postprocessor, len(nodes)


def sentence_window_rag(
    query: str,
    retriever,
    postprocessor: MetadataReplacementPostProcessor,
) -> dict:
    """Full Sentence Window pipeline: retrieve sentences → expand windows → generate.

    Args:
        query:          The user's question.
        retriever:      Sentence-level retriever from the built index.
        postprocessor:  MetadataReplacementPostProcessor for window expansion.

    Returns:
        dict with keys: query, sentence_nodes, window_nodes, answer, latency_ms.
    """
    t0 = time.perf_counter()

    # Step 1: retrieve top-k sentence nodes by sentence-level embedding similarity
    sentence_nodes = retriever.retrieve(query)

    # Step 2: expand each matched sentence to its ±window_size surrounding sentences
    # MetadataReplacementPostProcessor swaps node.text with node.metadata['window']
    window_nodes = postprocessor.postprocess_nodes(sentence_nodes)

    # Step 3: build generation context from window-expanded nodes
    context = '\n\n---\n\n'.join(
        f'[score={node.score:.4f}]\n{node.get_content()}'
        for node in window_nodes
    )

    # Step 4: generate answer — model receives window context, not just sentences
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system=(
            'Answer using only the provided contract context. '
            'Cite specific clause language where available.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )

    return {
        'query'         : query,
        'sentence_nodes': sentence_nodes,   # pre-expansion: matched sentences
        'window_nodes'  : window_nodes,     # post-expansion: window context
        'answer'        : response.content[0].text.strip(),
        'latency_ms'    : (time.perf_counter() - t0) * 1000,
    }


# Build the sentence-level index once; reuse retriever across queries
retriever, postprocessor, node_count = build_sentence_index(raw_doc)

print(f'Sentence index built: {node_count} sentence nodes')
print(f'  Each node embeds one sentence + stores ±{WINDOW_SIZE} window in metadata')
print('Pipeline ready: sentence_window_rag(query, retriever, postprocessor)')

## Cell 4: Run — Margin Call Trigger Query
**What this demonstrates**: A precise legal query that benefits from sentence-level retrieval. The margin call trigger is a single sentence; the ±3 window provides the Threshold Amount definition and cure period that make the answer complete.
**Expected output**: Top retrieved sentence with its score, the expanded window, and the final answer citing specific clause language.

In [ ]:
QUERY = 'What triggers a margin call?'

print(f'Query: {QUERY}')
print()

result = sentence_window_rag(QUERY, retriever, postprocessor)

# Show the top matched sentence (pre-expansion)
top_sentence = result['sentence_nodes'][0]
original_sentence = top_sentence.node.metadata.get('original_text', top_sentence.node.get_content())
print('Top matched sentence (retrieval unit):')
print(f'  Score: {top_sentence.score:.4f}')
print(f'  Text : {original_sentence.strip()}')
print()

# Show the expanded window (generation unit)
top_window = result['window_nodes'][0]
print('Expanded window (generation unit):')
print('-' * 60)
print(top_window.get_content()[:500])
if len(top_window.get_content()) > 500:
    print('  ... [truncated]')
print('-' * 60)
print()

print(f'Retrieval summary:')
print(f'  Sentences retrieved : {len(result["sentence_nodes"])}')
print(f'  Windows after expand: {len(result["window_nodes"])}')
print(f'  Latency             : {result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])

## Cell 5: Inspect — Exact Sentence, Expanded Window, Citation Trail
**What this demonstrates**: The mechanics of sentence-level precision — showing the exact matched sentence for each retrieved node, its surrounding window, the expansion delta (characters added by the window), and a baseline comparison against standard chunk retrieval.
**Expected output**: Per-node sentence vs window comparison, expansion metrics, and baseline retrieval contrast.

In [ ]:
from llama_index.core import VectorStoreIndex as _VI
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document as LCDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Per-node: sentence vs window ───────────────────────────────────────────────
print('SENTENCE vs WINDOW EXPANSION (per retrieved node)')
print('=' * 70)
for i, (snode, wnode) in enumerate(zip(result['sentence_nodes'], result['window_nodes']), 1):
    original = snode.node.metadata.get('original_text', snode.node.get_content()).strip()
    window   = wnode.get_content().strip()
    expansion_chars = len(window) - len(original)
    print(f'\nNode {i} | score={snode.score:.4f}')
    print(f'  Matched sentence ({len(original)} chars):')
    print(f'    {original[:120]}...' if len(original) > 120 else f'    {original}')
    print(f'  Expanded window  ({len(window)} chars, +{expansion_chars} from window):')
    print(f'    {window[:200]}...' if len(window) > 200 else f'    {window}')

# ── Citation trail ─────────────────────────────────────────────────────────────
print()
print('CITATION TRAIL — top matched sentence with metadata')
print('=' * 70)
top = result['sentence_nodes'][0]
print(f'  Source   : {top.node.metadata.get("source", "isda_excerpt")}')
print(f'  Score    : {top.score:.4f}')
original_top = top.node.metadata.get('original_text', top.node.get_content()).strip()
print(f'  Sentence : {original_top}')
print(f'  Window   : {top.node.metadata.get("window", "")[:300]}...')

# ── Baseline: standard chunk retrieval ────────────────────────────────────────
print()
print('BASELINE: standard 400-char chunk retrieval on same query')
print('=' * 70)

# Build a standard chunk index for comparison
_lc_embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
_lc_doc = LCDocument(page_content=text, metadata={'source': 'isda_excerpt'})
_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
_chunks = _splitter.split_documents([_lc_doc])
_chunk_store = Chroma.from_documents(_chunks, _lc_embeddings, collection_name='isda_baseline')
_baseline_results = _chunk_store.similarity_search(QUERY, k=3)

print(f'Chunk index: {len(_chunks)} chunks (400-char)')
print(f'Sentence index: {node_count} sentence nodes')
print()
print('Top chunk returned (first 300 chars):')
print(_baseline_results[0].page_content[:300])
print()
print('Top sentence returned:')
print(result["sentence_nodes"][0].node.metadata.get('original_text', '').strip())
print()
print('Observation: chunk retrieval returns a paragraph averaging multiple clauses.')
print('Sentence retrieval returns the exact trigger sentence.')

## Cell 6: Fintech — Precise Contract Clause Lookup
**What this demonstrates**: Sentence Window Retrieval on a specific contract clause lookup — the kind of precise legal query common in derivatives operations, credit risk teams, and contract review workflows. The matched sentence provides a citeable reference; the window provides the interpretive context for a complete answer.
**Expected output**: Exact matched clause sentence, ±3 window context, and an answer that cites the precise contractual language.

In [ ]:
CLAUSE_QUERY = 'What are the conditions for early termination of the agreement?'

print('FINTECH: PRECISE CONTRACT CLAUSE LOOKUP')
print(f'Query: {CLAUSE_QUERY}')
print()
print('Use case: derivatives operations reviewing termination rights')
print('  → Need the exact clause sentence for a compliance checklist')
print('  → Need surrounding sentences for the conditional context')
print()

clause_result = sentence_window_rag(CLAUSE_QUERY, retriever, postprocessor)

# Show all retrieved sentences with scores
print('Retrieved sentences (precision layer):')
print('-' * 65)
for i, node in enumerate(clause_result['sentence_nodes'], 1):
    original = node.node.metadata.get('original_text', node.node.get_content()).strip()
    print(f'{i}. [score={node.score:.4f}]')
    print(f'   {original[:150]}...' if len(original) > 150 else f'   {original}')
    print()

# Show the top window expansion
print('Top window expansion (generation layer):')
print('-' * 65)
top_window_text = clause_result['window_nodes'][0].get_content().strip()
print(top_window_text[:600])
if len(top_window_text) > 600:
    print('  ... [truncated]')
print()

# Expansion metrics
top_sentence_text = clause_result['sentence_nodes'][0].node.metadata.get(
    'original_text', clause_result['sentence_nodes'][0].node.get_content()
).strip()
print(f'Sentence length : {len(top_sentence_text)} chars')
print(f'Window length   : {len(top_window_text)} chars')
print(f'Context added   : {len(top_window_text) - len(top_sentence_text)} chars from ±{WINDOW_SIZE} window')
print(f'Latency         : {clause_result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('CONTRACT CLAUSE ANSWER')
print('=' * 65)
print(clause_result['answer'])
print()
print('-' * 65)
print('Sentence Window value for contract work:')
print('  Matched sentence = citeable reference for compliance trail')
print('  Window context   = conditions, definitions, consequences')
print('  Together: precise citation + interpretable answer')